> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Effective tests on a bounded HS genotype panel

VIGOR 2.12.1. The panel result is not a calibrated threshold for the entire genome.


In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__


In [ ]:
from vigor.gwas import effective_tests
train=pd.read_csv(TRAIN_CSV,index_col=0);train.index=train.index.astype(str)
geno=vigor.open_genotype_dataset(PLINK_PREFIX,source_format="plink1",variant_chunk=4096)
# Select a bounded panel before materializing. Full-genome calibration needs its own scope.
panel=geno.GT.sel(sample=train.index.astype(str)).isel(variant=slice(0,1000)).compute()
rows=[]
for method in ("ld_iact","ld_decay","block_spectral"):
    result=effective_tests(panel.values,method=method,cache=False)
    rows.append({"method":method,"meff":result.meff})
display(pd.DataFrame(rows))
